# 05 — RAG Chat Logic

Where retrieval meets reasoning: take a question, pull the most relevant facts from Notebook 4's index, hand them to an LLM with strict instructions to use *only* that context, and return an answer with its sources — "The Voice" from the original pitch.

**On the model choice:** a hosted API rather than Ollama, specifically because of where this project runs. Ollama needs its own local server process and multi-gigabyte model weights — fine on a personal machine, but Colab sessions are ephemeral, so you'd re-download those weights every single time a session restarts. A hosted API has a small per-call cost, but no reinstall tax and far more reliable answers. This notebook uses Anthropic's Claude API; swapping to OpenAI is a small, contained change if you'd rather (just the `call_llm` function below).

In [ ]:
!pip install anthropic -q

## Reload the index and facts

Notebook 4 already built and saved the FAISS index — no need to rebuild it, just load it back along with the embedding model (to encode new questions) and `facts.csv` (the lookup table).

In [ ]:
import pandas as pd
import faiss
import anthropic
from sentence_transformers import SentenceTransformer
from getpass import getpass

DATA_DIR = "../data"
facts_df = pd.read_csv(f"{DATA_DIR}/facts.csv")
index = faiss.read_index(f"{DATA_DIR}/facts.faiss")
embed_model = SentenceTransformer("all-MiniLM-L6-v2")
print(f"Loaded {len(facts_df)} facts and an index of {index.ntotal} vectors")

## API key

Entered through `getpass` rather than typed into a cell, so it never ends up saved in the notebook file or committed to GitHub by accident.

In [ ]:
api_key = getpass("Enter your Anthropic API key: ")
client = anthropic.Anthropic(api_key=api_key)

## Retrieval (same logic as Notebook 4)

In [ ]:
MIN_SCORE = 0.35

def search(query, k=5, min_score=MIN_SCORE):
    q_vec = embed_model.encode([query], convert_to_numpy=True).astype("float32")
    faiss.normalize_L2(q_vec)
    scores, ids = index.search(q_vec, k)
    return [
        {"text": facts_df.iloc[i]["text"], "category": facts_df.iloc[i]["category"],
         "n": int(facts_df.iloc[i]["n"]), "score": float(s)}
        for i, s in zip(ids[0], scores[0]) if s >= min_score
    ]

## The prompt

The system instruction is the whole reason this is "RAG" rather than just an LLM guessing: it's told to answer *only* from the facts it's given, and to say so honestly when the facts don't fully cover the question, rather than filling gaps with plausible-sounding invention.

In [ ]:
SYSTEM_PROMPT = (
    "You are a career advisor for software developers. Answer using ONLY the survey "
    "facts provided in the context below — never use outside knowledge or assumptions. "
    "Each fact states its sample size (n); weigh facts from larger samples more heavily, "
    "and mention when a number rests on a small sample. If the provided facts don't fully "
    "answer the question, say so plainly rather than guessing at the rest."
)

def build_user_message(question, retrieved_facts):
    context = "\n".join(f"- {f['text']}" for f in retrieved_facts)
    return f"Context (2025 Stack Overflow Developer Survey):\n{context}\n\nQuestion: {question}"

## Tying it together

In [ ]:
def ask(question, k=5):
    retrieved = search(question, k=k)
    if not retrieved:
        return {
            "answer": "I don't have survey data that speaks to that directly — I can only answer from the 2025 Stack Overflow Developer Survey facts I've indexed.",
            "sources": [],
        }
    user_message = build_user_message(question, retrieved)
    response = client.messages.create(
        model="claude-haiku-4-5-20251001",
        max_tokens=500,
        system=SYSTEM_PROMPT,
        messages=[{"role": "user", "content": user_message}],
    )
    return {"answer": response.content[0].text, "sources": retrieved}


def show(question):
    result = ask(question)
    print(f"Q: {question}\n")
    print(f"A: {result['answer']}\n")
    if result["sources"]:
        print("Sources:")
        for s in result["sources"]:
            print(f"  [{s['category']}, n={s['n']}] {s['text'][:80]}")
    print("\n" + "="*80 + "\n")

## Test it — starting with your own example from the pitch

In [ ]:
show("I'm a junior dev in India learning Python. Should I learn Rust or Go next for the best salary bump?")

In [ ]:
show("Is a CS degree worth it compared to being self-taught?")
show("What is the best pizza topping?")

## What we built

- Retrieval and generation fully wired together: a question becomes a search, the results become grounded context, and the LLM answers strictly from that context
- The out-of-scope test should show the honest "no data" response rather than a hallucinated guess — worth double-checking when you run this for real
- Every answer comes with its sources shown, matching the "Sources Cited" UX from the original pitch

**One thing I couldn't verify from here:** the actual `client.messages.create` call needs a real API key and a live network call, neither of which I can do from this sandbox. The code follows the current, documented Anthropic SDK pattern, but you'll be the one to see the real answers come back.

**Next:** Notebook 5 lives in a notebook — the last step is porting this same `ask()` logic into `app.py`, a standalone Streamlit script, for the actual chat interface.